#  E3: Construindo um Agente SINARM do Zero

**MBA IA Generativa PCDF - IBMEC**  
**Encontro 3:** Construção Hands-On

---

##  Objetivo

Criar ferramentas (tools) que consultam dados do SINARM.

**O que vamos construir:**
1.  Primeira tool simples
2.  Decorators (@tool, @lru_cache)
3.  4 tools especializadas
4.  Roteador inteligente
5.  Validação e segurança

**Tempo estimado:** 3-4 horas

**Nota:** Esta versão foca em tools e decorators. Integração com LLM será no E4.

---

##  PASSO 1: Instalação e Imports

**O que vamos fazer:**
- Instalar dependências necessárias
- Importar bibliotecas
- Verificar se tudo está OK

In [1]:
# Instalar dependências (executar apenas uma vez)
# Descomente as linhas abaixo se necessário:

# !pip install langchain-core==0.3.28
# !pip install pandas==2.2.3

In [2]:
# Imports necessários
import pandas as pd
from functools import lru_cache
from langchain_core.tools import tool

print(" Imports carregados com sucesso!")

 Imports carregados com sucesso!


---

##  PASSO 2: Carregar Dados SINARM

**O que vamos fazer:**
- Criar função para carregar dados
- Usar `@lru_cache` para otimizar (cache)
- Visualizar primeiras linhas

**Conceito:** `@lru_cache` guarda resultado na memória para não recarregar sempre.

In [3]:
# Funcao para carregar CSV (com cache)
@lru_cache(maxsize=1)
def carregar_csv():
    """Carrega dados do SINARM com cache"""
    print("Carregando CSV...")
    
    # Dados sinteticos para demonstracao - distribuicao realista
    marcas = []
    calibres = []
    tipos = []
    
    # TAURUS: 30 registros
    marcas.extend(['TAURUS'] * 30)
    calibres.extend(['9mm'] * 12 + ['.38'] * 10 + ['.40'] * 5 + ['.380'] * 3)
    tipos.extend(['FURTO'] * 12 + ['ROUBO'] * 10 + ['APREENSAO'] * 7 + ['PERDA'] * 1)
    
    # GLOCK: 25 registros
    marcas.extend(['GLOCK'] * 25)
    calibres.extend(['9mm'] * 15 + ['.40'] * 8 + ['.45'] * 2)
    tipos.extend(['ROUBO'] * 15 + ['FURTO'] * 7 + ['APREENSAO'] * 3)
    
    # BERETTA: 20 registros
    marcas.extend(['BERETTA'] * 20)
    calibres.extend(['9mm'] * 12 + ['.40'] * 5 + ['.380'] * 3)
    tipos.extend(['APREENSAO'] * 10 + ['FURTO'] * 7 + ['ROUBO'] * 3)
    
    # IMBEL: 15 registros
    marcas.extend(['IMBEL'] * 15)
    calibres.extend(['.38'] * 10 + ['9mm'] * 5)
    tipos.extend(['FURTO'] * 8 + ['APREENSAO'] * 5 + ['ROUBO'] * 2)
    
    # ROSSI: 12 registros
    marcas.extend(['ROSSI'] * 12)
    calibres.extend(['.38'] * 8 + ['.357'] * 4)
    tipos.extend(['FURTO'] * 6 + ['ROUBO'] * 4 + ['PERDA'] * 2)
    
    # SIG SAUER: 10 registros
    marcas.extend(['SIG SAUER'] * 10)
    calibres.extend(['9mm'] * 6 + ['.40'] * 4)
    tipos.extend(['ROUBO'] * 6 + ['FURTO'] * 3 + ['APREENSAO'] * 1)
    
    # SMITH & WESSON: 8 registros
    marcas.extend(['SMITH & WESSON'] * 8)
    calibres.extend(['.38'] * 5 + ['.357'] * 3)
    tipos.extend(['FURTO'] * 4 + ['ROUBO'] * 3 + ['APREENSAO'] * 1)
    
    df = pd.DataFrame({
        'MARCA_ARMA': marcas,
        'CALIBRE': calibres,
        'TIPO_OCORRENCIA': tipos
    })
    
    print(f"OK - {len(df)} registros carregados!")
    return df

print("OK - Funcao carregar_csv() criada com cache!")

OK - Funcao carregar_csv() criada com cache!


In [4]:
# Verificar distribuicao dos dados
# IMPORTANTE: Execute a celula anterior (carregar_csv) antes desta!

df = carregar_csv()

print("Distribuicao por Marca:")
print(df['MARCA_ARMA'].value_counts())

print("\nDistribuicao por Calibre:")
print(df['CALIBRE'].value_counts())

print("\nDistribuicao por Tipo:")
print(df['TIPO_OCORRENCIA'].value_counts())

Carregando CSV...
OK - 120 registros carregados!
Distribuicao por Marca:
MARCA_ARMA
TAURUS            30
GLOCK             25
BERETTA           20
IMBEL             15
ROSSI             12
SIG SAUER         10
SMITH & WESSON     8
Name: count, dtype: int64

Distribuicao por Calibre:
CALIBRE
9mm     50
.38     33
.40     22
.357     7
.380     6
.45      2
Name: count, dtype: int64

Distribuicao por Tipo:
TIPO_OCORRENCIA
FURTO        47
ROUBO        43
APREENSAO    27
PERDA         3
Name: count, dtype: int64


In [5]:
df = carregar_csv()

# Verificar distribuição dos dados
print(" Distribuição por Marca:")
print(df['MARCA_ARMA'].value_counts())

print("\n Distribuição por Calibre:")
print(df['CALIBRE'].value_counts())

print("\n Distribuição por Tipo:")
print(df['TIPO_OCORRENCIA'].value_counts())

 Distribuição por Marca:
MARCA_ARMA
TAURUS            30
GLOCK             25
BERETTA           20
IMBEL             15
ROSSI             12
SIG SAUER         10
SMITH & WESSON     8
Name: count, dtype: int64

 Distribuição por Calibre:
CALIBRE
9mm     50
.38     33
.40     22
.357     7
.380     6
.45      2
Name: count, dtype: int64

 Distribuição por Tipo:
TIPO_OCORRENCIA
FURTO        47
ROUBO        43
APREENSAO    27
PERDA         3
Name: count, dtype: int64


---

##  PASSO 3: Primeira Tool Simples

**O que vamos fazer:**
- Criar nossa primeira ferramenta: contar armas por marca
- Usar decorator `@tool` do LangChain
- Testar a tool diretamente

**Conceito:** `@tool` transforma função Python em ferramenta que pode ser usada por agentes.

In [6]:
@tool
def contar_armas_marca(marca: str) -> str:
    """Conta quantas armas de uma marca específica estão registradas.
    
    Args:
        marca: Nome da marca (ex: TAURUS, GLOCK, BERETTA)
    
    Returns:
        String com a quantidade encontrada
    """
    df = carregar_csv()
    resultado = df[df["MARCA_ARMA"] == marca.upper()]
    return f"Encontrei {len(resultado)} armas {marca}"

# Testar tool
print(" Testando tool:")
print(contar_armas_marca.invoke({"marca": "TAURUS"}))
print(contar_armas_marca.invoke({"marca": "GLOCK"}))
print(contar_armas_marca.invoke({"marca": "BERETTA"}))

 Testando tool:
Encontrei 30 armas TAURUS
Encontrei 25 armas GLOCK
Encontrei 20 armas BERETTA


###  Entendendo o @tool

O decorator `@tool` faz 3 coisas importantes:

1. **Transforma** função Python em ferramenta LangChain
2. **Extrai** documentação (docstring) automaticamente
3. **Valida** tipos de entrada/saída

**Sem @tool:**
```python
resultado = contar_armas_marca("TAURUS")  # Chamada normal
```

**Com @tool:**
```python
resultado = contar_armas_marca.invoke({"marca": "TAURUS"})  # Chamada via invoke
```

---

##  PASSO 4: Mais Tools (Calibre, Tipo, Combinado)

**O que vamos fazer:**
- Criar 3 tools adicionais
- Tool para calibre
- Tool para tipo de ocorrência
- Tool combinada (marca + tipo)

In [7]:
@tool
def contar_armas_calibre(calibre: str) -> str:
    """Conta armas por calibre.
    
    Args:
        calibre: Calibre da arma (ex: 9mm, .38, .40, .45)
    
    Returns:
        String com a quantidade encontrada
    """
    df = carregar_csv()
    resultado = df[df["CALIBRE"].str.contains(calibre, case=False, na=False)]
    total = len(resultado)
    
    if total > 0:
        calibre_real = resultado["CALIBRE"].iloc[0]
        return f"Encontrei {total} armas calibre '{calibre_real}'"
    else:
        return f"Nao encontrei armas calibre '{calibre}'"


@tool
def contar_armas_tipo(tipo: str) -> str:
    """Conta por tipo de ocorrencia.
    
    Args:
        tipo: Tipo de ocorrencia (ex: FURTO, ROUBO, APREENSAO)
    
    Returns:
        String com a quantidade encontrada
    """
    df = carregar_csv()
    resultado = df[df["TIPO_OCORRENCIA"].str.contains(tipo.upper(), case=False, na=False)]
    total = len(resultado)
    
    if total > 0:
        tipo_real = resultado["TIPO_OCORRENCIA"].iloc[0]
        return f"Encontrei {total} ocorrencias tipo '{tipo_real}'"
    else:
        return f"Nao encontrei ocorrencias tipo '{tipo}'"


@tool
def contar_armas_combinado(marca: str, tipo: str) -> str:
    """Conta por marca E tipo de ocorrencia.
    
    Args:
        marca: Nome da marca
        tipo: Tipo de ocorrencia
    
    Returns:
        String com a quantidade encontrada
    """
    df = carregar_csv()
    resultado = df[
        (df["MARCA_ARMA"].str.contains(marca.upper(), case=False, na=False)) & 
        (df["TIPO_OCORRENCIA"].str.contains(tipo.upper(), case=False, na=False))
    ]
    total = len(resultado)
    
    if total > 0:
        marca_real = resultado["MARCA_ARMA"].iloc[0]
        tipo_real = resultado["TIPO_OCORRENCIA"].iloc[0]
        return f"Encontrei {total} armas '{marca_real}' tipo '{tipo_real}'"
    else:
        return f"Nao encontrei armas '{marca}' tipo '{tipo}'"

print("OK - 4 tools criadas (com busca parcial)!")

OK - 4 tools criadas (com busca parcial)!


In [8]:
# Testar todas as tools
print(" Testando todas as tools:\n")

print("1. Por marca:")
print(contar_armas_marca.invoke({"marca": "GLOCK"}))

print("\n2. Por calibre:")
print(contar_armas_calibre.invoke({"calibre": "9mm"}))

print("\n3. Por tipo:")
print(contar_armas_tipo.invoke({"tipo": "FURTO"}))

print("\n4. Combinado:")
print(contar_armas_combinado.invoke({"marca": "TAURUS", "tipo": "ROUBO"}))

 Testando todas as tools:

1. Por marca:
Encontrei 25 armas GLOCK

2. Por calibre:
Encontrei 50 armas calibre '9mm'

3. Por tipo:
Encontrei 47 ocorrencias tipo 'FURTO'

4. Combinado:
Encontrei 10 armas 'TAURUS' tipo 'ROUBO'


---

##  PASSO 4.5: Tools de Ranking e Estatísticas (NOVAS!)

**O que vamos fazer:**
- Criar 3 tools adicionais para análise
- `ranking_marcas()` - TOP 5 marcas mais registradas
- `ranking_calibres()` - TOP 5 calibres mais comuns
- `estatisticas_gerais()` - Resumo completo do banco

**Por que são úteis:**
- Respondem perguntas como "Qual marca tem mais registros?"
- Fornecem visão geral dos dados
- Complementam as tools básicas

In [9]:
@tool
def ranking_marcas(top_n: int = 5) -> str:
    """Retorna ranking das marcas mais registradas.
    
    Args:
        top_n: Número de marcas no ranking (padrão: 5)
    
    Returns:
        Lista das top N marcas com quantidades
    """
    df = carregar_csv()
    ranking = df["MARCA_ARMA"].value_counts().head(top_n)
    
    resultado = f"TOP {top_n} MARCAS MAIS REGISTRADAS:\n"
    for i, (marca, qtd) in enumerate(ranking.items(), 1):
        resultado += f"  {i} - {marca}: {qtd} armas\n"
    
    return resultado.strip()


@tool
def ranking_calibres(top_n: int = 5) -> str:
    """Retorna ranking dos calibres mais comuns.
    
    Args:
        top_n: Número de calibres no ranking (padrão: 5)
    
    Returns:
        Lista dos top N calibres com quantidades
    """
    df = carregar_csv()
    ranking = df["CALIBRE"].value_counts().head(top_n)
    
    resultado = f"TOP {top_n} CALIBRES MAIS COMUNS:\n"
    for i, (calibre, qtd) in enumerate(ranking.items(), 1):
        resultado += f"  {i} - {calibre}: {qtd} armas\n"
    
    return resultado.strip()


@tool
def estatisticas_gerais() -> str:
    """Retorna estatísticas gerais do banco de dados.
    
    Returns:
        Resumo completo: total de registros, marcas, calibres, tipos
    """
    df = carregar_csv()
    
    total_registros = len(df)
    total_marcas = df["MARCA_ARMA"].nunique()
    total_calibres = df["CALIBRE"].nunique()
    total_tipos = df["TIPO_OCORRENCIA"].nunique()
    
    marca_mais_comum = df["MARCA_ARMA"].value_counts().index[0]
    calibre_mais_comum = df["CALIBRE"].value_counts().index[0]
    tipo_mais_comum = df["TIPO_OCORRENCIA"].value_counts().index[0]
    
    resultado = f"""ESTATISTICAS GERAIS DO SINARM:

TOTAIS:
  - Registros: {total_registros}
  - Marcas diferentes: {total_marcas}
  - Calibres diferentes: {total_calibres}
  - Tipos de ocorrencia: {total_tipos}

MAIS COMUNS:
  - Marca: {marca_mais_comum}
  - Calibre: {calibre_mais_comum}
  - Tipo: {tipo_mais_comum}
"""
    return resultado.strip()

print(" 3 novas tools criadas! Total: 7 tools")

 3 novas tools criadas! Total: 7 tools


In [10]:
# Testar as novas tools
print(" Testando novas tools:\n")

print("1. Ranking de marcas:")
print(ranking_marcas.invoke({"top_n": 3}))

print("\n2. Ranking de calibres:")
print(ranking_calibres.invoke({"top_n": 3}))

print("\n3. Estatísticas gerais:")
print(estatisticas_gerais.invoke({}))

 Testando novas tools:

1. Ranking de marcas:
TOP 3 MARCAS MAIS REGISTRADAS:
  1 - TAURUS: 30 armas
  2 - GLOCK: 25 armas
  3 - BERETTA: 20 armas

2. Ranking de calibres:
TOP 3 CALIBRES MAIS COMUNS:
  1 - 9mm: 50 armas
  2 - .38: 33 armas
  3 - .40: 22 armas

3. Estatísticas gerais:
ESTATISTICAS GERAIS DO SINARM:

TOTAIS:
  - Registros: 120
  - Marcas diferentes: 7
  - Calibres diferentes: 6
  - Tipos de ocorrencia: 4

MAIS COMUNS:
  - Marca: TAURUS
  - Calibre: 9mm
  - Tipo: FURTO


---

## PASSO 4.6: Tool de Distribuicao (NOVA!)

**O que vamos fazer:**
- Criar tool para mostrar distribuicao de uma marca por tipo
- Responde perguntas como "Beretta em ocorrencias"
- Mostra percentuais de cada tipo

**Por que e util:**
- Analise detalhada por marca
- Visualizacao de padroes
- Complementa as tools basicas

In [11]:
@tool
def distribuicao_marca_por_tipo(marca: str) -> str:
    """Mostra distribuicao de uma marca por tipo de ocorrencia.
    
    Args:
        marca: Nome da marca
    
    Returns:
        Distribuicao detalhada por tipo de ocorrencia
    """
    df = carregar_csv()
    resultado = df[df["MARCA_ARMA"].str.contains(marca.upper(), case=False, na=False)]
    
    if len(resultado) == 0:
        return f"Nao encontrei armas da marca '{marca}'"
    
    marca_real = resultado["MARCA_ARMA"].iloc[0]
    distribuicao = resultado["TIPO_OCORRENCIA"].value_counts()
    
    resposta = f"DISTRIBUICAO DE {marca_real} POR TIPO:\n"
    resposta += f"  Total: {len(resultado)} armas\n\n"
    for tipo, qtd in distribuicao.items():
        percentual = (qtd/len(resultado)*100)
        resposta += f"  - {tipo}: {qtd} armas ({percentual:.1f}%)\n"
    
    return resposta.strip()

print("OK - Tool distribuicao_marca_por_tipo criada! Total: 8 tools")

OK - Tool distribuicao_marca_por_tipo criada! Total: 8 tools


In [12]:
# Testar a nova tool
print("Testando distribuicao:\n")

print("1. Beretta por tipo:")
print(distribuicao_marca_por_tipo.invoke({"marca": "Beretta"}))

print("\n2. Taurus por tipo:")
print(distribuicao_marca_por_tipo.invoke({"marca": "Taurus"}))

Testando distribuicao:

1. Beretta por tipo:
DISTRIBUICAO DE BERETTA POR TIPO:
  Total: 20 armas

  - APREENSAO: 10 armas (50.0%)
  - FURTO: 7 armas (35.0%)
  - ROUBO: 3 armas (15.0%)

2. Taurus por tipo:
DISTRIBUICAO DE TAURUS POR TIPO:
  Total: 30 armas

  - FURTO: 12 armas (40.0%)
  - ROUBO: 10 armas (33.3%)
  - APREENSAO: 7 armas (23.3%)
  - PERDA: 1 armas (3.3%)


---

##  PASSO 5: Criar Roteador Inteligente

**O que vamos fazer:**
- Criar função que analisa a pergunta
- Decidir qual tool usar
- Chamar a tool apropriada

**Conceito:** Roteador simples baseado em palavras-chave (sem LLM por enquanto)

In [13]:
def processar_pergunta(pergunta: str) -> str:
    """Processa pergunta e chama tool apropriada"""
    pergunta_lower = pergunta.lower()
    
    # PRIORIDADE 1: Ranking e estatisticas
    keywords_ranking_marca = ['ranking', 'top', 'mais registr', 'marca com mais', 'qual marca']
    if any(kw in pergunta_lower for kw in keywords_ranking_marca) and 'marca' in pergunta_lower:
        return ranking_marcas.invoke({"top_n": 5})
    
    keywords_ranking_calibre = ['ranking', 'top', 'mais comum', 'calibre com mais', 'qual calibre']
    if any(kw in pergunta_lower for kw in keywords_ranking_calibre) and 'calibre' in pergunta_lower:
        return ranking_calibres.invoke({"top_n": 5})
    
    keywords_estatisticas = ['estatistica', 'resumo', 'total', 'geral', 'quantos registros']
    if any(kw in pergunta_lower for kw in keywords_estatisticas):
        return estatisticas_gerais.invoke({})
    
    # PRIORIDADE 2: Detectar marca (com variacoes)
    marcas_map = {
        'taurus': ['taurus', 'tauros', 'tauru'],
        'glock': ['glock', 'glok'],
        'beretta': ['beretta', 'bereta'],
        'imbel': ['imbel', 'imbell'],
        'sig sauer': ['sig sauer', 'sig', 'sauer'],
        'rossi': ['rossi', 'rosi'],
        'smith': ['smith', 'smith & wesson', 'wesson']
    }
    
    marca_encontrada = None
    for marca_oficial, variacoes in marcas_map.items():
        for variacao in variacoes:
            if variacao in pergunta_lower:
                marca_encontrada = marca_oficial
                break
        if marca_encontrada:
            break
    
    # PRIORIDADE 3: Detectar calibre
    calibres = ['9mm', '.38', '.40', '.45', '.380', '380']
    calibre_encontrado = None
    for calibre in calibres:
        if calibre in pergunta_lower:
            calibre_encontrado = calibre
            break
    
    # PRIORIDADE 4: Detectar tipo
    tipos = ['furto', 'roubo', 'apreens', 'perda', 'roubad', 'furtad']
    tipo_encontrado = None
    for tipo in tipos:
        if tipo in pergunta_lower:
            if 'roub' in tipo:
                tipo_encontrado = 'roubo'
            elif 'furt' in tipo:
                tipo_encontrado = 'furto'
            elif 'apreens' in tipo:
                tipo_encontrado = 'apreensao'
            else:
                tipo_encontrado = tipo
            break
    
    # PRIORIDADE 5: Detectar perguntas sobre distribuicao
    keywords_distribuicao = ['distribuicao', 'por tipo', 'em ocorrencias', 'por ocorrencia', 'tipos de ocorrencia']
    if marca_encontrada and any(kw in pergunta_lower for kw in keywords_distribuicao):
        return distribuicao_marca_por_tipo.invoke({"marca": marca_encontrada})
    
    # DECISAO: Qual tool usar?
    if marca_encontrada and tipo_encontrado:
        return contar_armas_combinado.invoke({"marca": marca_encontrada, "tipo": tipo_encontrado})
    elif marca_encontrada:
        return contar_armas_marca.invoke({"marca": marca_encontrada})
    elif calibre_encontrado:
        return contar_armas_calibre.invoke({"calibre": calibre_encontrado})
    elif tipo_encontrado:
        return contar_armas_tipo.invoke({"tipo": tipo_encontrado})
    else:
        return "Nao consegui entender a pergunta. Tente perguntar sobre:\n  - Marca especifica (ex: 'Quantas Taurus?')\n  - Calibre (ex: 'Quantas 9mm?')\n  - Tipo de ocorrencia (ex: 'Quantos roubos?')\n  - Distribuicao (ex: 'Beretta por tipo')\n  - Ranking (ex: 'Top 5 marcas')\n  - Estatisticas gerais"

print("OK - Roteador inteligente criado (com suporte a 8 tools)!")

OK - Roteador inteligente criado (com suporte a 8 tools)!


In [14]:
# Testar roteador
print(" Testando roteador:\n")

print("1. Pergunta por marca:")
print(processar_pergunta("Quantas armas Glock?"))

print("\n2. Pergunta por calibre:")
print(processar_pergunta("Quantas armas calibre 9mm?"))

print("\n3. Pergunta por tipo:")
print(processar_pergunta("Quantas armas foram roubadas?"))

print("\n4. Pergunta combinada:")
print(processar_pergunta("Quantas armas Taurus foram roubadas?"))

 Testando roteador:

1. Pergunta por marca:
Encontrei 25 armas glock

2. Pergunta por calibre:
Encontrei 50 armas calibre '9mm'

3. Pergunta por tipo:
Encontrei 43 ocorrencias tipo 'ROUBO'

4. Pergunta combinada:
Encontrei 10 armas 'TAURUS' tipo 'ROUBO'


---

##  PASSO 6: Adicionar Validação e Segurança

**O que vamos fazer:**
- Criar função de validação
- Bloquear queries perigosas
- Criar função segura para perguntas

**Conceito:** Sempre validar entrada do usuário!

In [15]:
def validar_input(texto: str):
    """Valida entrada do usuário"""
    if len(texto) > 500:
        raise ValueError("Query muito longa")
    if len(texto) < 3:
        raise ValueError("Query muito curta")
    
    perigosos = [";", "--", "DROP", "DELETE", "UPDATE", "INSERT"]
    for char in perigosos:
        if char in texto.upper():
            raise ValueError(f"Caractere perigoso: {char}")
    
    return True


def perguntar_seguro(pergunta: str):
    """Faz pergunta com validação"""
    try:
        validar_input(pergunta)
        return processar_pergunta(pergunta)
    except ValueError as e:
        return f" ERRO: {e}"
    except Exception as e:
        return f" ERRO inesperado: {e}"

print(" Validação e segurança implementadas!")

 Validação e segurança implementadas!


In [16]:
# Testar validação
print(" Testando validação:\n")

print("1. Pergunta válida:")
print(perguntar_seguro("Quantas armas Beretta?"))

print("\n2. Pergunta com caractere perigoso (deve bloquear):")
print(perguntar_seguro("DROP TABLE armas"))

print("\n3. Pergunta muito curta (deve bloquear):")
print(perguntar_seguro("ok"))

 Testando validação:

1. Pergunta válida:
Encontrei 20 armas beretta

2. Pergunta com caractere perigoso (deve bloquear):
 ERRO: Caractere perigoso: DROP

3. Pergunta muito curta (deve bloquear):
 ERRO: Query muito curta


---

##  PASSO 7: Modo Interativo

**O que vamos fazer:**
- Criar loop de perguntas
- Permitir múltiplas perguntas
- Comando para sair

**Nota:** Execute a célula abaixo e faça perguntas!

In [17]:
print("="*70)
print("AGENTE SINARM v1.0 - MODO INTERATIVO")
print("="*70)
print("\nExemplos de perguntas:")
print("  - Quantas armas Taurus?")
print("  - Quantas armas calibre 9mm?")
print("  - Quantas armas Glock foram roubadas?")
print("\nDigite 'sair' para encerrar\n")
print("="*70 + "\n")

# Contador de perguntas
contador = 0

while True:
    pergunta = input("Sua pergunta: ")
    
    if pergunta.lower() in ['sair', 'exit', 'quit']:
        print("\n" + "="*70)
        print(f"Total de perguntas realizadas: {contador}")
        print("Ate logo!")
        print("="*70)
        break
    
    if not pergunta.strip():
        continue
    
    contador += 1
    resposta = perguntar_seguro(pergunta)
    
    # Exibir pergunta e resposta formatadas
    print("\n" + "="*70)
    print(f"PERGUNTA #{contador}:")
    print(f"  {pergunta}")
    print("\nRESPOSTA:")
    print(f"  {resposta}")
    print("="*70 + "\n")

AGENTE SINARM v1.0 - MODO INTERATIVO

Exemplos de perguntas:
  - Quantas armas Taurus?
  - Quantas armas calibre 9mm?
  - Quantas armas Glock foram roubadas?

Digite 'sair' para encerrar



PERGUNTA #1:
  qual a quantidade de armas da marca tauros registradas ?

RESPOSTA:
  Encontrei 30 armas taurus


PERGUNTA #2:
  qual a quantidade de armas tauros em ocorrencias ?

RESPOSTA:
  DISTRIBUICAO DE TAURUS POR TIPO:
  Total: 30 armas

  - FURTO: 12 armas (40.0%)
  - ROUBO: 10 armas (33.3%)
  - APREENSAO: 7 armas (23.3%)
  - PERDA: 1 armas (3.3%)


PERGUNTA #3:
  equipamentos da marca beretta registrados ?

RESPOSTA:
  Encontrei 20 armas beretta


PERGUNTA #4:
  equipamentos de marca berreta em ocorrencias ?

RESPOSTA:
  Nao consegui entender a pergunta. Tente perguntar sobre:
  - Marca especifica (ex: 'Quantas Taurus?')
  - Calibre (ex: 'Quantas 9mm?')
  - Tipo de ocorrencia (ex: 'Quantos roubos?')
  - Distribuicao (ex: 'Beretta por tipo')
  - Ranking (ex: 'Top 5 marcas')
  - Estatisticas g

---

## CONCLUSAO

**Parabens!** Voce construiu um sistema completo com:

- **8 ferramentas especializadas** (@tool)
- **Cache de dados** (@lru_cache)
- **Roteador inteligente** (analise de perguntas)
- **Validacao de seguranca** (bloqueia queries perigosas)
- **Modo interativo** (loop de perguntas)

---

## TOOLS IMPLEMENTADAS (8 total):

**Basicas (4):**
1. contar_armas_marca - Conta por marca especifica
2. contar_armas_calibre - Conta por calibre
3. contar_armas_tipo - Conta por tipo de ocorrencia
4. contar_armas_combinado - Conta marca + tipo

**Avancadas (4):**
5. ranking_marcas - TOP 5 marcas mais registradas
6. ranking_calibres - TOP 5 calibres mais comuns
7. estatisticas_gerais - Resumo completo dos dados
8. distribuicao_marca_por_tipo - Distribuicao de marca por tipo

---

## PROXIMOS PASSOS

1. **E4 - RAG:** Adicionar busca semantica para perguntas conceituais
2. **LLM:** Integrar com Ollama para roteamento mais inteligente
3. **Dados reais:** Substituir dados sinteticos por CSV real do SINARM
4. **Mais tools:** Graficos, analises avancadas, exportacao de relatorios

---

## USAR EM PRODUCAO

**Arquivo pronto para uso:**
```
03_AGENTE_CONSOLIDADO/agente_sinarm_v2_completo.py
```

**Como executar:**
```bash
cd ../03_AGENTE_CONSOLIDADO
python agente_sinarm_v2_completo.py
```

**Status:** 100% funcional e testado (11/11 testes aprovados)

---

**Desenvolvido no MBA IA Generativa PCDF - IBMEC**  
**E3 - Construcao do Agente do Zero**